# 5.3 更新方向怎麼累積？


同一格參數連續收到梯度1、1，第三次變-1。只聽當輪梯度會立即反向；保留先前方向的一部分，則可能再向原方向走一步。這種跨輪累積叫動量。

用v保存方向、w保存位置；g是當輪梯度，μ是保留比例，η是學習率。規則是新v＝μ×舊v＋g，再令新w＝舊w−η×新v。以下μ=0.9、η=0.1，兩個狀態都從0開始；三個梯度是我們指定的材料。


In [1]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [2]:
velocity = 0.0
position = 0.0
for gradient in [1.0, 1.0, -1.0]:
    velocity = 0.9 * velocity + gradient
    position -= 0.1 * velocity
    print("新梯度", gradient, "保留方向", round(velocity, 3), "位置", round(position, 3))

新梯度 1.0 保留方向 1.0 位置 -0.1
新梯度 1.0 保留方向 1.9 位置 -0.29
新梯度 -1.0 保留方向 0.71 位置 -0.361


輸入是我們指定的三個梯度1、1、-1，不是從模型測出的資料。v與w最初都0。第一步v=1，位置-0.1；第二步v=0.9+1=1.9，位置-0.29；第三步雖新梯度為-1，v仍為 `0.9×1.9-1=0.71`，位置繼續下降到-0.361。打印的三列把當前意見、歷史合成與真正參數移動分開，`round` 只是方便顯示。

第三步最值得觀察：當前代價的梯度已轉向，動量卻暫時延續先前方向。這可能幫助穿過小抖動，也可能讓參數越過谷底後還前進過頭。因而動量並不保證每一步的當下代價都下降，要與學習率一起看訓練曲線。μ靠近1時記憶較長，同時反轉也較慢。

這裡沒有定義代價函數，不能從位置越來越負判斷品質。每格參數各有自己的 v，續訓需保存它。動量也要與[1.13](https://birdhackor.github.io/tiny-perceptron-vlm/1.13.html)的梯度累加分清：`backward()` 沿運算關係算當次梯度，再加進參數的 `.grad`。若在一次更新前多次呼叫而沒有清空 `.grad`，會把本輪多份梯度相加；本節的 v 則在更新後仍保留，讓過去方向繼續影響下一輪。

練習只把保留係數0.9改成0，先手算三位置為-0.1、-0.2、-0.1，再執行核對。第三步立即反向，因為沒有過去推力。最後試0.5，應得到v為1、1.5、-0.25：同樣的梯度序列，保留強度決定什麼時候轉彎。

<details>
<summary>補充：實作約定與原始紀錄</summary>

短程式沿用本課工具與原計算語義。安裝、長訓練與重做操作見[訓練配方](https://birdhackor.github.io/tiny-perceptron-vlm/training.html)，不需要先完成長配方才能閱讀這個例子。

</details>
